In [1]:
%load_ext autoreload
%autoreload 2
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import normflows as nf
import formula as fo
import summarize as su
import getplot as pl
import time
from torch.distributions import Normal

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [33]:
class NBase(nn.Module):
    def __init__(self, dim, init_loc=0.0, init_log_scale=-2.5):
        super().__init__()
        self.dim = dim
        self.loc = nn.Parameter(torch.full((dim,), float(init_loc)))
        self.raw_log_scale = nn.Parameter(torch.full((dim,), float(init_log_scale)))

    def log_scale(self):
        return torch.clamp(self.raw_log_scale, min=-5.0, max=2.0)

    def scale(self):
        return torch.exp(self.log_scale())

    def rsample(self, num_samples):
        eta = torch.randn(
            num_samples,
            self.dim,
            device=self.loc.device,
            dtype=self.loc.dtype,
        )
        z0 = self.loc.unsqueeze(0) + self.scale().unsqueeze(0) * eta
        return eta, z0

    def log_prob(self, z0):
        log_scale = self.log_scale().unsqueeze(0)
        var = torch.exp(2.0 * log_scale)
        return -0.5 * (
            ((z0 - self.loc.unsqueeze(0)) ** 2) / var
            + 2.0 * log_scale
            + math.log(2.0 * math.pi)
        ).sum(dim=1)

In [34]:
def simfun1(n=180, p=100, seed=123, snr=3.0, true_prop=0.1, device=None, dtype=torch.float32,):

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    X = rng.standard_normal((n, p)).astype(np.float32)
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)

    n_active = int(p * true_prop)
    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))

    beta_true = np.zeros(p, dtype=np.float32)
    magnitudes = rng.uniform(0.3, 2.0, size=n_active).astype(np.float32)
    signs = rng.choice([-1.0, 1.0], size=n_active).astype(np.float32)
    beta_true[active_idx] = signs * magnitudes

    signal = X @ beta_true
    sigma2 = np.var(signal) / snr
    sigma = np.sqrt(sigma2)

    y = signal + sigma * rng.standard_normal(n).astype(np.float32)
    y = y - y.mean()

    X_t = torch.tensor(X, dtype=dtype, device=device)
    y_t = torch.tensor(y, dtype=dtype, device=device)
    beta_true_t = torch.tensor(beta_true, dtype=dtype, device=device)

    info = {"n": n, "p": p, "n_active": n_active, "sigma2": float(sigma2), "sigma": float(sigma), "active_idx": active_idx, "snr": snr,}

    return X_t, y_t, beta_true_t, info

In [35]:
class AffineCoupling(nn.Module):
    """
    Handwritten masked affine coupling layer,
    but using normflows' MLP as the internal network.
    """
    def __init__(self, dim, mask, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        self.dim = dim
        self.register_buffer("mask", mask.view(1, dim))
        self.scale_clip = float(scale_clip)

        widths = [dim] + [hidden_units] * num_hidden_layers + [dim]
        self.s_net = nf.nets.MLP(widths, init_zeros=True)
        self.t_net = nf.nets.MLP(widths, init_zeros=True)

    def forward(self, x, return_logdet=False):
        x_masked = x * self.mask

        raw_log_scale = self.s_net(x_masked) * (1.0 - self.mask)
        shift = self.t_net(x_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        y = x_masked + (1.0 - self.mask) * (x * torch.exp(log_scale) + shift)

        if return_logdet:
            logdet = log_scale.sum(dim=-1)
            return y, logdet
        return y

    def inverse(self, y, return_logdet=False):
        y_masked = y * self.mask

        raw_log_scale = self.s_net(y_masked) * (1.0 - self.mask)
        shift = self.t_net(y_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        x = y_masked + (1.0 - self.mask) * ((y - shift) * torch.exp(-log_scale))

        if return_logdet:
            logdet = (-log_scale).sum(dim=-1)
            return x, logdet
        return x


class FlowMap(nn.Module):
    """
    Same structure as your original TransportMap:
    alternate odd/even masks, stack K affine coupling layers.
    """
    def __init__(self, dim, K=4, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        self.dim = dim

        base_mask = torch.tensor(
            [1.0 if i % 2 == 0 else 0.0 for i in range(dim)],
            dtype=torch.float32,
        )

        layers = []
        for k in range(K):
            mask = base_mask if (k % 2 == 0) else (1.0 - base_mask)
            layers.append(
                AffineCoupling(
                    dim=dim,
                    mask=mask,
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                )
            )

        self.layers = nn.ModuleList(layers)

    def forward(self, x, return_logdet=False):
        z = x
        if return_logdet:
            total_logdet = x.new_zeros(x.shape[0])

        for layer in self.layers:
            if return_logdet:
                z, logdet = layer(z, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                z = layer(z)

        if return_logdet:
            return z, total_logdet
        return z

    def inverse(self, z, return_logdet=False):
        x = z
        if return_logdet:
            total_logdet = z.new_zeros(z.shape[0])

        for layer in reversed(self.layers):
            if return_logdet:
                x, logdet = layer.inverse(x, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                x = layer.inverse(x)

        if return_logdet:
            return x, total_logdet
        return x

In [36]:
class Relaxedsas(nn.Module):
    def __init__(self, X, y, sigma2, tau, g_theta):
        super().__init__()
        if g_theta is None:
            raise ValueError("g_theta must be provided.")

        self.register_buffer("X", X)
        self.register_buffer("y", y)
        self.register_buffer("sigma2", torch.tensor(float(sigma2), dtype=X.dtype, device=X.device))
        self.register_buffer("tau", torch.tensor(float(tau), dtype=X.dtype, device=X.device))

        self.n, self.p = X.shape
        self.dim = 2 * self.p + 1
        self.g_theta = g_theta

    def set_tau(self, tau):
        self.tau.fill_(float(tau))

    def decode(self, eps):
        xi = self.g_theta(eps)

        p = self.p
        s = xi[:, :p]
        u = xi[:, p:2 * p]
        t = xi[:, 2 * p:2 * p + 1]

        gate = torch.sigmoid((u - t) / self.tau)
        beta = s * gate

        return {"eps": eps, "xi": xi, "s": s, "u": u, "t": t, "gate": gate, "beta": beta,}

    def log_joint(self, eps):
        dec = self.decode(eps)
        beta = dec["beta"]

        fit = self.X @ beta.T
        resid = self.y[:, None] - fit

        loglik = -0.5 * (
            resid.pow(2).sum(dim=0) / self.sigma2
            + self.n * torch.log(2.0 * torch.pi * self.sigma2)
        )

        log_p0_eps = -0.5 * (
            eps.pow(2) + math.log(2.0 * math.pi)
        ).sum(dim=1)

        return loglik + log_p0_eps

In [37]:
class FlowVI(nn.Module):
    def __init__(self, q0, posterior_flow, generative_model):
        super().__init__()
        self.q0 = q0
        self.posterior_flow = posterior_flow
        self.generative_model = generative_model

    def sample_posterior(self, num_samples):
        _, z0 = self.q0.rsample(num_samples)
        eps, logdet = self.posterior_flow(z0, return_logdet=True)
        log_q_eps = self.q0.log_prob(z0) - logdet
        return eps, log_q_eps

    def neg_elbo(self, num_samples=256, elbo_beta=1.0):
        eps, log_q_eps = self.sample_posterior(num_samples)
        log_joint = self.generative_model.log_joint(eps)
        return (log_q_eps - elbo_beta * log_joint).mean()


def build_flow_vi(X, y, sigma2, tau, K_q=8, K_g=8, hidden_units=64, num_hidden_layers=2,):
    dim = 2 * X.shape[1] + 1

    g_theta = FlowMap(
        dim=dim,
        K=K_g,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=2.0,
    )

    generative_model = Relaxedsas(X=X, y=y, sigma2=sigma2, tau=tau, g_theta=g_theta,)

    q0 = NBase(dim=dim, init_loc=0.0, init_log_scale=-2.5,)
    
    posterior_flow = FlowMap(
        dim=dim,
        K=K_q,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=2.0,
    )

    return FlowVI(q0=q0, posterior_flow=posterior_flow, generative_model=generative_model,)

In [38]:
def train_flow(model, epochs=1000, num_samples=128, lr=5e-5, tau_start=1.0, tau_end=0.05,
    anneal_ratio=0.7, grad_clip=3.0, print_every=100, elbo_beta=1.0,):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    tau_hist = []
    grad_hist = []
    
    anneal_epochs = max(1, int(round(epochs * anneal_ratio)))

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        if epoch <= anneal_epochs:
            frac = (epoch - 1) / max(anneal_epochs - 1, 1)
            tau_now = tau_start * (tau_end / tau_start) ** frac
        else:
            tau_now = float(tau_end)

        model.generative_model.set_tau(tau_now)
        tau_hist.append(float(tau_now))

        loss = model.neg_elbo(num_samples=num_samples, elbo_beta=elbo_beta)
        loss.backward()

        if grad_clip is not None:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            grad_norm = float(grad_norm)
        else:
            grad_norm = 0.0

        optimizer.step()

        losses.append(float(loss.item()))
        grad_hist.append(grad_norm)
        
        with torch.no_grad():
            point_trace = trace_eps_point(model)
            u = point_trace["u_hat"].squeeze(0)
            t = point_trace["t_hat"].squeeze(0)
            marg = u - t
            n_selected = int((marg > 0).sum().item())
            near_boundary = int((marg.abs() < 0.05).sum().item())

        
        if epoch % print_every == 0 or epoch == 1:
            print(
                f"[epoch {epoch:04d}] "
                f"loss={loss.item():.6f} "
                f"tau={tau_now:.6f} "
                f"grad_norm={grad_norm:.6f} "
                f"n_selected={n_selected} "
                f"near_boundary={near_boundary}"
            )

    return losses, tau_hist, grad_hist

In [39]:
@torch.no_grad()
def trace_eps_point(model):
    z0_hat = model.q0.loc.unsqueeze(0)
    eps_hat, _ = model.posterior_flow(z0_hat, return_logdet=True)
    dec = model.generative_model.decode(eps_hat)

    return {
        "s_hat": dec["s"].detach().cpu(),
        "u_hat": dec["u"].detach().cpu(),
        "t_hat": dec["t"].detach().cpu(),
        "beta_hat": dec["beta"].detach().cpu(),
    }

@torch.no_grad()
def point_selection_summary(point_trace, beta_true):
    u_hat = point_trace["u_hat"].squeeze(0)
    t_hat = point_trace["t_hat"].squeeze(0)
    beta_hat = point_trace["beta_hat"].squeeze(0)

    beta_true_cpu = beta_true.detach().cpu()

    pred = (u_hat > t_hat).int().numpy()
    truth = (beta_true_cpu.abs() > 1e-12).int().numpy()

    tp = int(((pred == 1) & (truth == 1)).sum())
    fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum())
    tn = int(((pred == 0) & (truth == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    if t_hat.numel() == 1:
        t_col = np.repeat(t_hat.item(), u_hat.numel())
    else:
        t_col = t_hat.numpy()

    df = pd.DataFrame({"j": np.arange(u_hat.numel()), "beta_true": beta_true_cpu.numpy(),
        "u_hat": u_hat.numpy(), "t_hat": t_col, "beta_hat": beta_hat.numpy(),
        "truth": truth, "pred": pred,
    })

    metrics = {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1,
        "n_selected": int(pred.sum()), "selected_idx": np.where(pred == 1)[0].tolist(),
    }

    return df, metrics

In [42]:
def simflow(seed=123, device=None, dtype=torch.float32, hidden_units=64, num_hidden_layers=2,
    n=180, p=100, snr=3.0, true_prop=0.1, tau_end=0.5, K_q=8, K_g=8,
    epochs=1000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=100,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X, y, beta_true, sim_info = simfun1(n=n, p=p, seed=seed, snr=snr, true_prop=true_prop, device=device, dtype=dtype,)

    print("===== Simulation info =====")
    print(sim_info)

    model = build_flow_vi(X=X, y=y, sigma2=sim_info["sigma2"], tau=tau_end, K_q=K_q, K_g=K_g,
        hidden_units=hidden_units, num_hidden_layers=num_hidden_layers,
    ).to(device)

    losses, tau_hist, grad_hist = train_flow(model=model, epochs=epochs, num_samples=num_samples,
        lr=lr, tau_start=1.0, tau_end=tau_end, anneal_ratio=1, grad_clip=grad_clip, print_every=print_every, elbo_beta=1.0,
    )

    point_trace = trace_eps_point(model)
    selection_df, metrics = point_selection_summary(point_trace, beta_true)

    return {"beta_true": beta_true, "sim_info": sim_info, "model": model, "losses": losses,
        "tau_hist": tau_hist, "grad_hist": grad_hist, "point_trace": point_trace,
        "selection_df": selection_df, "metrics": metrics,
    }

In [ ]:
out_flow = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1,
    tau_end=0.18, K_q=16, K_g=16, hidden_units=64, num_hidden_layers=2,
    epochs=4000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=891.764465 tau=1.000000 grad_norm=275.459290 n_selected=0 near_boundary=100
[epoch 0500] loss=253.303925 tau=0.798325 grad_norm=81.108047 n_selected=8 near_boundary=1
[epoch 1000] loss=245.280014 tau=0.637035 grad_norm=92.726479 n_selected=7 near_boundary=0
[epoch 1500] loss=243.858002 tau=0.508332 grad_norm=84.236977 n_selected=8 near_boundary=0
[epoch 2000] loss=243.548508 tau=0.405631 grad_norm=83.007980 n_selected=8 near_boundary=1
[epoch 2500] loss=243.332397 tau=0.323679 grad_norm=122.838722 n_selected=9 near_boundary=2
[epoch 3000] loss=243.319855 tau=0.258285 grad_norm=116.700386 n_selected=11 near_boundary=3
[epoch 3500] loss=243.185699 tau=0.206102 grad_norm=81.410042 n_selected=12 near_boundary=4
[epoch 4000] loss=243.109787 tau=0.180000 grad_norm=97.

In [32]:
print(out_flow["metrics"])
out_flow["selection_df"].head(15)

{'tp': 8, 'fp': 6, 'fn': 2, 'tn': 84, 'precision': 0.5714285714285714, 'recall': 0.8, 'f1': 0.6666666666666666, 'n_selected': 14, 'selected_idx': [5, 10, 14, 23, 24, 28, 29, 35, 37, 45, 64, 69, 90, 93]}


,j,beta_true,u_hat,t_hat,beta_hat,truth,pred
0,0,0.000000,-0.059145,0.18068,-0.392777,0,0
1,1,0.000000,0.052998,0.18068,-0.382253,0,0
2,2,0.000000,0.023569,0.18068,-0.794281,0,0
3,3,0.000000,-24.879959,0.18068,0.000000,0,0
4,4,0.000000,0.172090,0.18068,1.280276,0,0
5,5,0.000000,0.993342,0.18068,0.535751,0,1
6,6,0.000000,-0.190526,0.18068,-0.281345,0,0
7,7,0.000000,0.071071,0.18068,0.368069,0,0
8,8,0.000000,-0.337917,0.18068,-0.081637,0,0
9,9,0.000000,-0.005118,0.18068,0.388561,0,0


In [29]:
u = out_flow["point_trace"]["u_hat"].squeeze(0).numpy()
t = float(out_flow["point_trace"]["t_hat"].squeeze().item())
marg = u - t

print("t_hat =", t)
print("n_selected =", (marg > 0).sum())
print("margin quantiles =", np.quantile(marg, [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
print("near boundary =", np.sum(np.abs(marg) < 0.05))

t_hat = 0.026556676253676414
n_selected = 38
margin quantiles = [-6.49461528e+01 -1.83743189e+01 -1.08601667e-01 -1.73256779e-02
  3.07888547e-02  1.53103395e-01  3.42402459e+01]
near boundary = 43


In [43]:
out_flow = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1,
    tau_end=0.1, K_q=16, K_g=16, hidden_units=64, num_hidden_layers=2,
    epochs=6000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=891.764465 tau=1.000000 grad_norm=275.459290 n_selected=0 near_boundary=100
[epoch 0500] loss=253.247192 tau=0.825695 grad_norm=79.722534 n_selected=8 near_boundary=0
[epoch 1000] loss=245.010208 tau=0.681510 grad_norm=98.821831 n_selected=7 near_boundary=0
[epoch 1500] loss=243.508133 tau=0.562503 grad_norm=98.674271 n_selected=8 near_boundary=0
[epoch 2000] loss=243.303970 tau=0.464278 grad_norm=53.421093 n_selected=8 near_boundary=0
[epoch 2500] loss=243.096954 tau=0.383204 grad_norm=57.331394 n_selected=8 near_boundary=1
[epoch 3000] loss=243.061951 tau=0.316288 grad_norm=70.624458 n_selected=9 near_boundary=1
[epoch 3500] loss=242.929672 tau=0.261057 grad_norm=97.266609 n_selected=11 near_boundary=3
[epoch 4000] loss=242.868011 tau=0.215471 grad_norm=82.181